# Baseline Model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly as px
from scipy import stats

In [2]:
# Load the data
df_processed = pd.read_csv('')


In [3]:
# Shape
print(f'The dataframe has {df_processed.shape[0]} rows and {df_processed.shape[1]} columns.')


The dataframe has 534730 rows and 602 columns.


Given the large size of the dataset, this Notebook was built and tested using a 20% sample of the data to speed up the code and allow for quick verification before running the analysis on the entire dataset. The hyperparameter optimization results varied after the first iteration with the full dataset, which helped to fine-tune the parameters before running the final pipelines.

In [4]:
from sklearn.model_selection import train_test_split

# Split the dataset into sample and remainder sets, with stratification based on the class labels
df_processed, remainder_df = train_test_split(df_processed_original, test_size=0.8, stratify=df_processed_original['disposition'], random_state=1)



In [5]:
df_processed.shape

(534730, 602)

# Modeling

- Logistic Regression: commonly used model for binary classification problems. With a high number of columns there may be many potential interactions and non-linearities among the features. That being said, it's always worth trying logistic regression as a baseline model for comparison purposes, especially if the dataset has a small number of informative features. This model is simple, interpretable, and computationally efficient, and it can serve as a good starting point for model selection.

## Logistic Regression Model

Previous Assumptions:
- Observations are independent of each other.
- There is no perfect multicollinearity between the independent variables.
- There is a large enough sample size to provide reliable estimates of the coefficients.
- Should not be outliers in the data.
- The errors (residuals) are independent of each other.


In [6]:
# Data
df_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 534730 entries, 0 to 534729
Columns: 602 entries, disposition to whtblooddx
dtypes: float64(202), int64(400)
memory usage: 2.4 GB


In [7]:
# Define the predictor variables
X = df_processed.drop(columns=['inhospital_mortality'],axis=1).copy()

# Define the target variable
y = df_processed['inhospital_mortality']

In [8]:
# Import the library
from sklearn.model_selection import train_test_split

# Create the train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                 test_size=0.2, 
                                                 random_state=3, 
                                                 stratify=y)

In [9]:
# Logistic regression baseline model
from sklearn.linear_model import LogisticRegression

baseline_logreg = LogisticRegression(solver='lbfgs', max_iter=100000, random_state=1)
baseline_logreg.fit(X_train, y_train)
predictions_train = baseline_logreg.predict(X_train)
predictions_test = baseline_logreg.predict(X_test)
blg_train_accuracy = baseline_logreg.score(X_train, y_train)
blr_test_accuracy = baseline_logreg.score(X_test, y_test)

# Print the accuracy of the model
print(f"Baseline_logred train accuracy: {blg_train_accuracy}")
print(f"Baseline_logreg test accuracy: {blr_test_accuracy}")

Baseline_logred train accuracy: 0.85773861842158
Baseline_logreg test accuracy: 0.8553350912298419


It looks like the model is performing really well, ~86% accuracy. However, remember ~ 30% of patients were admitted and ~70% were discharge, imbalanced data forces us to rely on other evaluation metrics than accuracy: precision, recall, F1 score, and area under the ROC curve to evaluate our model.

**Predictors**

In [10]:
# Logistic Regression Coeficients (remove # too se the coeficients)
# baseline_logreg.coef_

In [11]:
# Define the feature_label and its coefficient
column_labels = X_train.columns.tolist()

# Squeeze function to convert the numpy array into a 1-dimensional array
coef = baseline_logreg.coef_.squeeze().tolist()

In [12]:
# Create a dictionary with column_labels as key and coeficients as values (remove '#' to visualize all the predictors)

lg_predictors = dict(zip(column_labels, coef))
# lg_predictors 

In [13]:
# Top 20 most predictive features by coefficient

sorted_predictors = sorted(lg_predictors.items(), key=lambda x: x[1], reverse=True)
sorted_predictors[:20]

[('esi_1', 2.534205657527735),
 ('cc_respiratorydistress', 1.9791669293153697),
 ('cc_strokealert', 1.3950927620058848),
 ('meds_analgesicandantihistaminecombination', 1.2642724651234627),
 ('cc_follow-upcellulitis', 1.2556146769561956),
 ('meds_blood', 1.2354893398707485),
 ('cc_feverimmunocompromised', 1.2345191189070754),
 ('cc_fever-75yearsorolder', 1.2090707508720426),
 ('cc_fulltrauma', 1.163598375145192),
 ('cc_cellulitis', 1.1517048933066296),
 ('cc_overdose-intentional', 1.1114522684743837),
 ('meds_contraceptives', 1.104714405258871),
 ('esi_2', 1.0208820132240823),
 ('cc_fever-9weeksto74years', 0.9893705863325243),
 ('cc_rectalbleeding', 0.9519777954735752),
 ('cc_neurologicproblem', 0.9511592951439044),
 ('cc_extremityweakness', 0.9460197956056103),
 ('acutphanm', 0.9206945013696161),
 ('cc_dehydration', 0.8967353036043028),
 ('cc_withdrawal-alcohol', 0.8794540882610489)]

The best predictor is...

**Model evaluation**

Lets see the confusion matrix to evaluate other scores:

In [14]:
# Import the confusion matrix module
from sklearn.metrics import confusion_matrix

# Generate confusion matrix
cf_matrix = confusion_matrix(y_test, predictions_test)

# label rows and columns
cf_df = pd.DataFrame(
    cf_matrix, 
    columns=["Predicted Discharge", "Predicted Admitted"],
    index=["True Discharge", "True Admitted"]
)

display(cf_df)

,Predicted Discharge,Predicted Admitted
True Discharge,104306,6696
True Admitted,16511,32906


**Results running the whole dataframe:**

...

In [15]:
# Import the classification_report module
from sklearn.metrics import classification_report

report = classification_report(y_test, predictions_test)
print(report)

              precision    recall  f1-score   support

           0       0.86      0.94      0.90    111002
           1       0.83      0.67      0.74     49417

    accuracy                           0.86    160419
   macro avg       0.85      0.80      0.82    160419
weighted avg       0.85      0.86      0.85    160419



Precision: the precision for class 0.0 is 0.86, which means that 86% of the instances that the model predicted as 0.0 were actually 0.0. Similarly, the precision for class 1.0 is 0.83, indicating that 83% of the instances that the model predicted as 1.0 were actually 1.0.

Recall: the recall for class 0.0 is 0.94, which means that 94% of the actual instances that were 0.0 were correctly predicted by the model. 

F1-score: F1-score for class 0.0 is 0.90, and the F1-score for class 1.0 is 0.74. The weighted average F1-score is 0.85, indicating that the model performed reasonably well in overall.

**The most important here is that recall for class 1.0 is 0.66, indicating that 66% of the actual instances that were 1.0 were correctly predicted by the model. Since that the priority is identify those patients that require be admitted faster it is really necessary correct the imbalance and improve the admitted patients predictions**.

## Class imbalance

To correct the imbalance in this case the approach is going to be downsampling. (A very important note here is that which ever approach will be applied should only be done to the training data, the test data should remain untouched and have class imbalance consistent with what you would expect the real world to be.)

### Downsampling

Downsampling is a method for reducing the size of a dataset by randomly removing observations from the majority class, which can help to balance the class distribution in a binary classification problem. This can be useful when the dataset is imbalanced and the minority class is underrepresented.

In [16]:
from sklearn.utils import resample

print('Number of class 0 examples before:', X_train[y_train == 0].shape[0])

X_downsampled, y_downsampled = resample(X_train[y_train == 0], 
                                        y_train[y_train == 0], 
                                        replace=True, 
                                        n_samples=X_train[y_train == 1].shape[0], 
                                        random_state=123)

print('Number of class 0 examples after:', X_downsampled.shape[0])

# Combine downsample with original data
X_train_bal2 = np.vstack((X_train[y_train == 1], X_downsampled))
y_train_bal2 = np.hstack((y_train[y_train == 1], y_downsampled))

# Convert the X variable numpy array into a dataframe
X_train_bal2 = pd.DataFrame(X_train_bal2, columns=X.columns)

Number of class 0 examples before: 259005
Number of class 0 examples after: 115306


Let's try again the logistic regression model using the downsampled data:

In [17]:
# Logistic regression after downsampling

resample_logreg = LogisticRegression(solver='lbfgs', max_iter=100000, random_state=1)
resample_logreg.fit(X_train_bal2, y_train_bal2)
predictions_train = resample_logreg.predict(X_train_bal2)
predictions_test_ds = resample_logreg.predict(X_test)
resample_train_accuracy = resample_logreg.score(X_train_bal2, y_train_bal2)
test_accuracy = resample_logreg.score(X_test, y_test)

# Print the accuracy of the model
print(f"Baseline_logred train accuracy: {resample_train_accuracy}")
print(f"Baseline_logreg test accuracy: {test_accuracy}")

Baseline_logred train accuracy: 0.8310625639602449
Baseline_logreg test accuracy: 0.8386787101278527


Quite similar accuracy but it is more relevant evaluate recall:

In [18]:
# Generate confusion matrix
cf_matrix = confusion_matrix(y_test, predictions_test_ds)

# label rows and columns
cf_df = pd.DataFrame(
    cf_matrix, 
    columns=["Predicted Discharge", "Predicted Admitted"],
    index=["True Discharge", "True Admitted"]
)

display(cf_df)

,Predicted Discharge,Predicted Admitted
True Discharge,94625,16377
True Admitted,9502,39915


In [19]:
# Logistic Regression Report with downsampling
report_ds = classification_report(y_test, predictions_test_ds)
print(report_ds)

              precision    recall  f1-score   support

           0       0.91      0.85      0.88    111002
           1       0.71      0.81      0.76     49417

    accuracy                           0.84    160419
   macro avg       0.81      0.83      0.82    160419
weighted avg       0.85      0.84      0.84    160419



Remember Logistic Regression Report without downsampling

In [20]:
# Logistic Regression Report without downsampling
report = classification_report(y_test, predictions_test)
print(report)

              precision    recall  f1-score   support

           0       0.86      0.94      0.90    111002
           1       0.83      0.67      0.74     49417

    accuracy                           0.86    160419
   macro avg       0.85      0.80      0.82    160419
weighted avg       0.85      0.86      0.85    160419



Now the model is generating more false admitted patients (Type I error) and less false discharged patients (Type II error), the recall improved but the precision went down. Gladly the overal performance is quite similar ~84%.